In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline
import joblib
from datasets import Dataset
from transformers import DataCollatorWithPadding
import torch


C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained("finiteautomata/bertweet-base-sentiment-analysis")
model = AutoModelForSequenceClassification.from_pretrained(
    "finiteautomata/bertweet-base-sentiment-analysis",
    id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'},
    label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7576.25it/s]


In [3]:
df = pd.read_csv("processed_sentiment_data.csv")

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

In [4]:
def tokenize_fn(batch):
  return tokenizer(
      batch["processed_text"],
      truncation = True,
      max_length= 128,
      padding = False
  )
tokenized_dataset = dataset.map(tokenize_fn, batched = True)
tokenized_dataset = tokenized_dataset.remove_columns(["processed_text"])
tokenized_dataset = tokenized_dataset.rename_column("sentiment", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer) #data collator forms batches using the dataset
train_ds = tokenized_dataset["train"].shuffle(seed=42).select(range(80000))
eval_ds  = tokenized_dataset["test"].shuffle(seed=42).select(range(20000))

Map: 100%|██████████| 21693/21693 [00:16<00:00, 1278.65 examples/s]


In [5]:
# !pip install -q evaluate scikit-learn
# !pip install -q -U torchvision

In [6]:
import evaluate
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

#https://deepwiki.com/huggingface/transformers/3.2-trainingarguments-configuration
#This website has a ton of information about the different parameters.
training_args = TrainingArguments(
    output_dir="./roberta-sentiment-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    #per_device_train_batch_size controls training batch size. per_device_eval_batch_size evaluation batch size.
    learning_rate=.00005,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    logging_steps=50,
    weight_decay=0.01, #this adds a L2 penalty
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, #half precision
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback()]
    #https://github.com/huggingface/transformers/blob/v5.14.0/src/transformers/trainer_callback.py#L711
    #https://huggingface.co/docs/transformers/v5.14.0/en/main_classes/callback#transformers.EarlyStoppingCallback
)
trainer.train()

C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.509014,0.790000,0.788212
2,0.626224,0.532621,0.830000,0.821394
3,0.361771,0.532852,0.790000,0.792887


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]
C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]
C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]


TrainOutput(global_step=24, training_loss=0.46261141697565716, metrics={'train_runtime': 5151.2195, 'train_samples_per_second': 0.777, 'train_steps_per_second': 0.006, 'total_flos': 197335063296000.0, 'train_loss': 0.46261141697565716, 'epoch': 3.0})

In [8]:
metrics = trainer.evaluate()
print(metrics)
from sklearn.metrics import classification_report
preds_output = trainer.predict(eval_ds)
y_pred = np.argmax(preds_output.predictions, axis =-1)
y_true = preds_output.label_ids
print(classification_report(y_true, y_pred))

C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.361771,0.532621,3,0.830000,0.821394


{'eval_loss': 0.5326208472251892, 'eval_accuracy': 0.83, 'eval_f1': 0.8213940632135561}


C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.86      0.86      0.86        95
           1       0.71      0.47      0.57        32
           2       0.82      0.95      0.88        73

    accuracy                           0.83       200
   macro avg       0.80      0.76      0.77       200
weighted avg       0.82      0.83      0.82       200



In [70]:
trainer.save_model("roberta_sentiment_model")
tokenizer.save_pretrained("roberta_sentiment_model")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.37s/it]


('roberta_sentiment_model\\tokenizer_config.json',
 'roberta_sentiment_model\\vocab.txt',
 'roberta_sentiment_model\\bpe.codes',
 'roberta_sentiment_model\\added_tokens.json')

In [4]:
#I didnt now how to do the actual predictions part and I wanted to find an easy way to use the tokenizer and model. Using this text classification pipeline makes it easy to combine it, instead of making a predict method.
#https://discuss.huggingface.co/t/i-have-trained-my-classifier-now-how-do-i-do-predictions/3625
roberta_model = TextClassificationPipeline(
    model = model,
    tokenizer = tokenizer,
)

In [5]:
from logreg_class import LogRegTfidfModel
from finetuned_roberta_class import FTRobertaModel

logreg_model = LogRegTfidfModel()
fine_tuned_model = FTRobertaModel()

def predict(text):
    logreg_model.predict(text)
    #logreg_model.print_prediction()

    fine_tuned_model.predict(text)
    #fine_tuned_model.print_prediction()

    r_pred = roberta_model.predict(text)
    state = 2
    if r_pred[0]['label'] == "Negative":
        state = 0
    elif r_pred[0]['label'] == "Neutral":
        state = 1
    # print(f"\n** Roberta Model **\nPredicted Sentiment: {r_pred[0]['label']}\nConfidence: %{(r_pred[0]['score'] * 100):.2f}")
    return logreg_model.pred_number, fine_tuned_model.pred_number, state


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sungj\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sungj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sungj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sungj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sungj\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\sungj\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nl

In [7]:
iters = len(df)
start_iter = 0
f_count = 0
r_count = 0
l_count = 0
too_long_idx = []


for i in np.arange(start_iter, start_iter+iters, 1):
    try:
        l_pred, f_pred, r_pred = predict(df.loc[i, "processed_text"])

        act = df.loc[i, "sentiment"]
        if f_pred == act:
            f_count += 1
        if r_pred == act:
            r_count += 1
        if l_pred == act:
            l_count += 1

    except RuntimeError:
        too_long_idx.append(i)

# df = df.drop(index=too_long_idx).reset_index(drop=True)
# df.to_csv("processed_sentiment_data.csv", index=False)
print(f"LogReg Accuracy: %{(l_count / (iters - len(too_long_idx)) * 100):.2f}")
print(f"Roberta Accuracy: %{(r_count / (iters - len(too_long_idx)) * 100):.2f}")
print(f"FineTuned Roberta Accuracy: %{(f_count / (iters - len(too_long_idx)) * 100):.2f}")
print(f"# of bad idxs: {len(too_long_idx)}")


LogReg Accuracy: %85.11
Roberta Accuracy: %72.83
FineTuned Roberta Accuracy: %90.44
# of bad idxs: 9352
